# Exercise 20.2: Chunking & Similarity Metrics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-20/02-chunking-and-similarity-metrics.ipynb)

- **Book Reference:** Chapter 20 (*Embeddings, Vector Search (RAG) and Fine-Tuning (LoRA)*, Section 20.1)
- **Estimated Time:** 15–20 minutes
- **Learning Objectives:**
  1. Implement sliding-window word chunking with configurable overlap for RAG preprocessing.
  2. Compare dot-product, cosine similarity, and Euclidean distance rankings on unnormalised vs. unit-normalised embeddings.
  3. Verify mathematically when all three metrics induce the exact same ranking.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

## Mathematical Background

When all embedding vectors are **unit-normalised** ($\lVert \mathbf{q} \rVert = \lVert \mathbf{d}_i \rVert = 1$):
- Dot product equals cosine similarity: $\mathbf{q} \cdot \mathbf{d}_i = \cos\theta_i$.
- Squared Euclidean distance is a strictly decreasing function of cosine similarity:
  $$\lVert \mathbf{q} - \mathbf{d}_i \rVert^2 = \lVert \mathbf{q} \rVert^2 + \lVert \mathbf{d}_i \rVert^2 - 2 \mathbf{q} \cdot \mathbf{d}_i = 2(1 - \cos\theta_i)$$

However, when document vectors have **different magnitudes**, dot product favours high-norm vectors and Euclidean distance penalises norm mismatches, so the three rankings can disagree!

In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int = 0) -> list[str]:
    """Split text into overlapping word chunks.

    Args:
        text: Input document string.
        chunk_size: Maximum number of words per chunk (> 0).
        overlap: Number of overlapping words between consecutive chunks (0 <= overlap < chunk_size).

    Returns:
        List of chunk strings.
    """
    # TODO: Split text on whitespace and slice into windows of size `chunk_size` with step `chunk_size - overlap`
    raise NotImplementedError("Complete this TODO")


def dot_similarity(query: np.ndarray, docs: np.ndarray) -> np.ndarray:
    """Compute dot-product similarity docs @ query."""
    # TODO: Return 1D array of dot products
    raise NotImplementedError("Complete this TODO")


def cosine_similarity_scores(query: np.ndarray, docs: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between query (d,) and docs (N, d)."""
    # TODO: Return 1D array of cosine similarities
    raise NotImplementedError("Complete this TODO")


def euclidean_distances(query: np.ndarray, docs: np.ndarray) -> np.ndarray:
    """Compute L2 (Euclidean) distance ||docs[i] - query||_2 for each row."""
    # TODO: Return 1D array of Euclidean distances
    raise NotImplementedError("Complete this TODO")


def compare_rankings(query: np.ndarray, docs: np.ndarray) -> dict[str, np.ndarray]:
    """Return sorted index arrays (best-first) for 'dot', 'cosine', and 'euclidean'."""
    # TODO: Sort 'dot' and 'cosine' descending, and 'euclidean' ascending
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- In `chunk_text`, iterate `start` with `range(0, len(words), chunk_size - overlap)` and stop once `start + chunk_size >= len(words)`.
- Remember: higher is better for `dot` and `cosine` (`np.argsort(-scores)`), while **lower** is better for `euclidean` (`np.argsort(dists)`).

</details>

In [ ]:
# 1. Test chunk_text
sample = "one two three four five six seven"
chunks = chunk_text(sample, chunk_size=3, overlap=1)
assert chunks == ["one two three", "three four five", "five six seven"], f"Got {chunks}"

# 2. Unnormalised embeddings: dot & euclidean can disagree with cosine
q = np.array([1.0, 0.0])
docs_unnorm = np.array([
    [1.0, 0.0],    # idx 0: cos=1.0, dot=1.0, dist=0.0
    [5.0, 2.0],    # idx 1: cos<1.0, dot=5.0 (highest dot!), dist>0
    [0.5, 0.0],    # idx 2: cos=1.0, dot=0.5, dist=0.5
])
r_unnorm = compare_rankings(q, docs_unnorm)
assert r_unnorm["dot"][0] == 1, "Unnormalised dot product favours large-norm vector idx 1"
assert r_unnorm["euclidean"][0] == 0, "Euclidean distance picks exact match idx 0"

# 3. Unit-normalised embeddings: all 3 metrics agree!
q_unit = rng.normal(size=12)
q_unit /= np.linalg.norm(q_unit)
docs_raw = rng.normal(size=(10, 12))
docs_unit = docs_raw / np.linalg.norm(docs_raw, axis=1, keepdims=True)

r_norm = compare_rankings(q_unit, docs_unit)
assert np.array_equal(r_norm["dot"], r_norm["cosine"])
assert np.array_equal(r_norm["cosine"], r_norm["euclidean"])
print("All checks passed! Normalised rankings are identical:", r_norm["cosine"])

## Think Deeper

1. Why do production vector databases often L2-normalise embeddings at ingestion time?
2. What trade-off does increasing `overlap` in `chunk_text` create between retrieval recall and index size?